# Exercice 25b (cloud) : Stable Diffusion 1.5, échantillonneurs, guidage, puis un LoRA

L'exercice 25a entraîne un petit modèle de diffusion sur des chiffres 28 × 28. Ici, on manipule un
vrai modèle de **diffusion latente** : Stable Diffusion 1.5 (environ 860 millions de paramètres pour le
U-Net, images 512 × 512, débruitage dans un espace latent 64 × 64 × 4). Trois parties :

1. **Inférence** : la même image avec l'échantillonneur DDIM en 50, 20, 10 et 5 pas ; puis le poids
   du **guidage sans classifieur** (1 ; 3 ; 7,5 ; 15).
2. **LoRA** : adapter le modèle à 1 000 photos de papillons de collection du Smithsonian
   (domaine public, CC0) en n'entraînant que de petites matrices de rang 4 ajoutées aux couches
   d'attention du U-Net (moins de 1 % de ses paramètres ; la cellule 9 affiche le nombre exact).
3. **Avant / après** : mêmes prompts, mêmes graines, LoRA désactivé puis activé.

**Perte optimisée** : la même qu'à l'exercice 25a, l'erreur quadratique entre le bruit ajouté et
le bruit prédit (ε-prédiction), mais sur les latents de l'autoencodeur et avec le texte en condition :
$\mathcal L = \mathbb E\,\lVert \varepsilon - \varepsilon_\theta(z_t, t, \text{texte})\rVert^2$.

Avant de lancer : *Exécution › Modifier le type d'exécution › GPU T4*, puis *Exécution › Tout
exécuter*. Seed 42. Calcul en **float16** (le T4 ne fait pas de bfloat16), optimiseur **AdamW 8 bits**
(bitsandbytes). Aucun token Hugging Face n'est nécessaire : le modèle et les données ne sont pas
restreints.

In [ ]:
# Cellule 1 : vérifier le GPU
!nvidia-smi
import os
import torch
TEST_LOCAL = os.environ.get("IA_25B_TEST") == "1"   # vérification du code en miniature sur CPU (README)
if not TEST_LOCAL and not torch.cuda.is_available():
    raise SystemExit("Pas de GPU : Exécution › Modifier le type d'exécution › GPU T4, puis relancer.")
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# Cellule 2 : dépendances épinglées (torch n'est PAS épinglé : on garde celui de l'image Colab)
!pip install -q diffusers==0.41.0 peft==0.21.2 bitsandbytes==0.50.2 accelerate==1.15.0 transformers==5.19.0 datasets==5.1.0

In [ ]:
# Cellule 3 : configuration
import math, random, time
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# COURT = True : 100 pas d'entraînement seulement, pour vérifier que tout tourne
COURT = False
config = dict(
    modele="stable-diffusion-v1-5/stable-diffusion-v1-5",   # licence creativeml-openrail-m, non restreint
    donnees="huggan/smithsonian_butterflies_subset",       # 1 000 photos, issues de ceyda/smithsonian_butterflies (CC0)
    resolution=512,
    rang=4,                     # rang des matrices LoRA (lora_alpha = rang : échelle 1)
    lr=1e-4,                    # valeur par défaut du script officiel train_text_to_image_lora.py
    taille_lot=2,
    n_pas=100 if COURT else 800,
    pas_inference=30,
    guidage=7.5,
    declencheur="a photo of a smsn butterfly specimen",   # « smsn » : mot rare qui servira de déclencheur
)
dispositif = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if dispositif.type == "cuda" else torch.float32
if TEST_LOCAL:   # miniature : petit pipeline de test, 3 pas, 64 pixels (ne vérifie que le code)
    config.update(modele="hf-internal-testing/tiny-stable-diffusion-torch", resolution=64,
                  taille_lot=1, n_pas=3, pas_inference=3)
print(config, dispositif, dtype)

In [ ]:
# Cellule 4 : charger Stable Diffusion 1.5 en float16
from diffusers import DDIMScheduler, StableDiffusionPipeline

# Le filtre de sécurité (safety_checker) livré avec le modèle reste actif : une image jugée
# problématique est remplacée par une image noire.
# variant="fp16" : fichiers déjà en float16 (≈ 2,7 Go au lieu de 5,5 Go à télécharger)
_variante = {"variant": "fp16"} if dtype == torch.float16 else {}
pipe = StableDiffusionPipeline.from_pretrained(config["modele"], dtype=dtype, **_variante)
# DDIM : échantillonneur déterministe, sans réentraînement (Song et al., 2020)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(dispositif)
pipe.set_progress_bar_config(disable=True)
n_unet = sum(p.numel() for p in pipe.unet.parameters())
print(f"U-Net : {n_unet / 1e6:.0f} M de paramètres ; latent : {pipe.unet.config.in_channels} canaux, "
      f"facteur de réduction du VAE : {2 ** (len(pipe.vae.config.block_out_channels) - 1)}")


def generer(prompt, graine, pas=None, guidage=None):
    # Même graine → même bruit de départ → comparaison honnête
    g = torch.Generator(device=dispositif).manual_seed(graine)
    return pipe(prompt, num_inference_steps=pas or config["pas_inference"],
                guidance_scale=config["guidage"] if guidage is None else guidage,
                generator=g, height=config["resolution"], width=config["resolution"]).images[0]


def planche(images, titres, nom):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, len(images), figsize=(3.2 * len(images), 3.6))
    for ax, im, t in zip(np.atleast_1d(axes), images, titres):
        ax.imshow(im); ax.set_title(t, fontsize=10); ax.axis("off")
    fig.tight_layout(); fig.savefig(nom, dpi=110); plt.show()

In [ ]:
# Cellule 5 : partie 1a, nombre de pas de DDIM (même prompt, même graine, guidage 7,5)
prompt = "a watercolor painting of a lighthouse on a cliff at sunset"
images, titres = [], []
for pas in ([50, 20, 10, 5] if not TEST_LOCAL else [3, 2]):
    t0 = time.perf_counter()
    images.append(generer(prompt, SEED, pas=pas))
    titres.append(f"DDIM {pas} pas ({time.perf_counter() - t0:.1f} s)")
    print(titres[-1])
planche(images, titres, "25b-ddim-pas.png")

In [ ]:
# Cellule 6 : partie 1b, poids du guidage sans classifieur (30 pas)
# guidance_scale = 1 : prédiction conditionnelle seule (pas de guidage) ; au-delà, on extrapole
# depuis la prédiction sans texte vers la prédiction avec texte.
images, titres = [], []
for w in ([1.0, 3.0, 7.5, 15.0] if not TEST_LOCAL else [1.0, 7.5]):
    images.append(generer(prompt, SEED, guidage=w))
    titres.append(f"guidage {w:g}".replace(".", ","))
planche(images, titres, "25b-guidage.png")

In [ ]:
# Cellule 7 : partie 3 (avant), le modèle d'origine face au mot déclencheur
prompts_test = [config["declencheur"],
                config["declencheur"] + ", blue and black wings",
                "a smsn butterfly specimen on a leaf"]
graines_test = [0, 1, 2]
avant = [generer(p, g) for p, g in zip(prompts_test, graines_test)]
planche(avant, [f"avant, graine {g}" for g in graines_test], "25b-avant.png")

In [ ]:
# Cellule 8 : données, 1 000 photos de papillons de collection (Smithsonian, CC0)
from datasets import load_dataset
from torchvision import transforms

ds = load_dataset(config["donnees"], split="train")
print(ds)
prep = transforms.Compose([
    transforms.Resize(config["resolution"], interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(config["resolution"]),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),     # pixels dans [−1, 1], comme attendu par le VAE
])
# Une seule légende pour toutes les images : le LoRA associe le mot « smsn » au style de ces photos
ids_texte = pipe.tokenizer(config["declencheur"], padding="max_length", truncation=True,
                           max_length=pipe.tokenizer.model_max_length, return_tensors="pt").input_ids

def lot(indices):
    x = torch.stack([prep(ds[int(i)]["image"].convert("RGB")) for i in indices])
    return x.to(dispositif, dtype=dtype)

planche([ds[i]["image"] for i in range(4)], [f"donnée {i}" for i in range(4)], "25b-donnees.png")

In [ ]:
# Cellule 9 : ajouter le LoRA au U-Net (recette du script officiel train_text_to_image_lora.py)
from peft import LoraConfig
from diffusers.training_utils import cast_training_params

unet, vae, encodeur_texte = pipe.unet, pipe.vae, pipe.text_encoder
for m in (unet, vae, encodeur_texte):
    m.requires_grad_(False)                 # tout est gelé…
unet.add_adapter(LoraConfig(r=config["rang"], lora_alpha=config["rang"], init_lora_weights="gaussian",
                            target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
cast_training_params(unet, dtype=torch.float32)   # …sauf les matrices LoRA, gardées en float32
unet.enable_gradient_checkpointing()       # économise la mémoire du T4 (15 Go)
entrainables = [p for p in unet.parameters() if p.requires_grad]
n_lora = sum(p.numel() for p in entrainables)
print(f"Paramètres LoRA entraînés : {n_lora / 1e6:.2f} M, soit {n_lora / n_unet:.2%} du U-Net")

if dispositif.type == "cuda":
    import bitsandbytes as bnb
    optimiseur = bnb.optim.AdamW8bit(entrainables, lr=config["lr"])   # états d'Adam sur 8 bits
else:
    optimiseur = torch.optim.AdamW(entrainables, lr=config["lr"])     # test local sur CPU
echelle = torch.amp.GradScaler("cuda", enabled=dispositif.type == "cuda")
ordonnanceur = pipe.scheduler   # mêmes ᾱ_t que l'entraînement d'origine (1 000 pas)

In [ ]:
# Cellule 10 : entraînement du LoRA (perte : MSE sur le bruit, comme l'exercice 25a)
import torch.nn.functional as F

gen = torch.Generator().manual_seed(SEED)
with torch.no_grad():
    etats_texte = encodeur_texte(ids_texte.to(dispositif))[0]
pertes = []
unet.train()
t0 = time.perf_counter()
for pas in range(config["n_pas"]):
    indices = torch.randint(len(ds), (config["taille_lot"],), generator=gen)
    with torch.no_grad():
        # 1. image → latent (512 × 512 × 3 → 64 × 64 × 4), mis à l'échelle comme à l'entraînement de SD
        z0 = vae.encode(lot(indices)).latent_dist.sample() * vae.config.scaling_factor
    bruit = torch.randn_like(z0)
    t = torch.randint(0, ordonnanceur.config.num_train_timesteps, (z0.shape[0],), device=dispositif)
    zt = ordonnanceur.add_noise(z0, bruit, t)          # √ᾱ_t·z0 + √(1 − ᾱ_t)·ε
    with torch.autocast(dispositif.type, dtype=torch.float16, enabled=dispositif.type == "cuda"):
        pred = unet(zt, t, etats_texte.expand(z0.shape[0], -1, -1)).sample
    perte = F.mse_loss(pred.float(), bruit.float())
    optimiseur.zero_grad(set_to_none=True)
    echelle.scale(perte).backward()
    echelle.unscale_(optimiseur)
    torch.nn.utils.clip_grad_norm_(entrainables, 1.0)
    echelle.step(optimiseur)
    echelle.update()
    pertes.append(perte.item())
    if pas % 50 == 0 or pas == config["n_pas"] - 1:
        print(f"pas {pas:4d} | perte (moyenne des 50 derniers) {np.mean(pertes[-50:]):.4f} | "
              f"{time.perf_counter() - t0:.0f} s", flush=True)
duree_entrainement = time.perf_counter() - t0
unet.eval()
print(f"Entraînement : {duree_entrainement / 60:.1f} min pour {config['n_pas']} pas")

In [ ]:
# Cellule 11 : courbe de perte et sauvegarde des poids LoRA (quelques Mo)
import matplotlib.pyplot as plt
from peft.utils import get_peft_model_state_dict
from diffusers.utils import convert_state_dict_to_diffusers

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(pertes, alpha=0.3, label="par pas")
k = 25
if len(pertes) > k:
    ax.plot(range(k - 1, len(pertes)), np.convolve(pertes, np.ones(k) / k, mode="valid"), label=f"moyenne sur {k} pas")
ax.set(xlabel="pas", ylabel="MSE sur le bruit", title="Entraînement du LoRA"); ax.legend()
fig.savefig("25b-perte.png", dpi=110); plt.show()

poids = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
StableDiffusionPipeline.save_lora_weights("lora-papillons", unet_lora_layers=poids)
taille = sum(os.path.getsize(os.path.join("lora-papillons", f)) for f in os.listdir("lora-papillons"))
print(f"LoRA sauvegardé dans lora-papillons/ : {taille / 1e6:.1f} Mo (U-Net complet en float16 : ≈ {2 * n_unet / 1e9:.1f} Go)")

In [ ]:
# Cellule 12 : partie 3 (après), mêmes prompts, mêmes graines, LoRA désactivé puis activé
unet.disable_adapters()
sans = [generer(p, g) for p, g in zip(prompts_test, graines_test)]
unet.enable_adapters()
avec = [generer(p, g) for p, g in zip(prompts_test, graines_test)]
planche(sans + avec, [f"sans LoRA, graine {g}" for g in graines_test] + [f"avec LoRA, graine {g}" for g in graines_test],
        "25b-avant-apres.png")
# Le LoRA change-t-il aussi ce qui n'a rien à voir avec les papillons ?
autre = "a watercolor painting of a lighthouse on a cliff at sunset"
unet.disable_adapters(); a = generer(autre, SEED)
unet.enable_adapters(); b = generer(autre, SEED)
planche([a, b], ["phare, sans LoRA", "phare, avec LoRA"], "25b-phare.png")

In [ ]:
# Cellule 13 : récupérer les résultats avant la fin de session (Colab ; sur Kaggle : onglet Output)
import shutil
shutil.make_archive("resultats-25b", "zip", ".", "lora-papillons")
fichiers = ["resultats-25b.zip"] + [f for f in os.listdir(".") if f.startswith("25b-") and f.endswith(".png")]
print(fichiers)
try:
    from google.colab import files
    for f in fichiers:
        files.download(f)
except ImportError:
    print("Hors Colab : fichiers laissés dans le dossier courant.")